In [ ]:
"""
rockfall_lstm_prototype.py

Requirements:
- python 3.8+
- pip install numpy pandas tensorflow scikit-learn matplotlib

This script:
1. Generates synthetic sequential sensor data.
2. Trains an LSTM classifier (Keras) to predict risk class for sequence windows.
3. Runs a realtime-style sliding-window inference loop.

Note: In a real deployment replace synthetic data with your sensor streams,
use model checkpointing, edge optimization (TFLite), and proper alert channels.
"""

import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, utils
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import time

# ---------------------------
# Configuration
# ---------------------------
SEQ_LEN = 30          # sequence window length (timesteps)
N_FEATURES = 3        # slope, vibration, rainfall
N_SAMPLES = 3000      # number of sequences for training
BATCH_SIZE = 64
EPOCHS = 20
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ---------------------------
# Utility: label rule for a sequence
# ---------------------------
def label_sequence(seq):
    """
    seq: array shape (SEQ_LEN, N_FEATURES)
    rule (heuristic synthetic ground truth):
    - IMMINENT: any timestep with slope>3.5 OR (vibration>8 and rainfall>30) OR rapid slope increase
    - CAUTION: average slope > 2 or total rainfall in seq > 120
    - SAFE: otherwise
    """
    slopes = seq[:, 0]
    vibs = seq[:, 1]
    rains = seq[:, 2]

    # condition checks
    if np.any(slopes > 3.5) or np.any((vibs > 8) & (rains > 30)):
        return 2  # IMMINENT
    if np.mean(slopes) > 2.0 or np.sum(rains) > 120:
        return 1  # CAUTION
    return 0  # SAFE

# ---------------------------
# Synthetic sequence generator
# ---------------------------
def generate_sequence():
    """
    Create one multivariate sequence with realistic variability.
    slope (mm/hr): base small drift + occasional spikes
    vibration: baseline noise + events
    rainfall: slow varying (storm segments)
    """
    seq = np.zeros((SEQ_LEN, N_FEATURES), dtype=float)
    base_slope = random.uniform(0, 1.0)  # baseline slope
    base_vib = random.uniform(0, 3.0)
    base_rain = random.uniform(0, 5.0)

    rain_event_start = random.choice([None] + list(range(0, SEQ_LEN)))
    spike_prob = random.random()

    for t in range(SEQ_LEN):
        # slope drift + occasional upward spike
        slope = base_slope + 0.02 * t + (random.gauss(0, 0.2))
        if spike_prob > 0.95 and random.random() > 0.7:
            slope += random.uniform(2.0, 4.5)  # spike
        # vibration noise and bursts
        vib = base_vib + random.gauss(0, 0.5)
        if random.random() > 0.98:
            vib += random.uniform(5, 10)
        # rainfall: slow changes, maybe storm occupying several timesteps
        if rain_event_start is not None and t >= rain_event_start and t < rain_event_start + random.randint(3, 10):
            base_rain += random.uniform(2.0, 10.0)
        rain = max(0.0, base_rain + random.gauss(0, 1.0))

        seq[t, 0] = slope
        seq[t, 1] = vib
        seq[t, 2] = rain

    return seq

def generate_dataset(n_samples):
    X = np.zeros((n_samples, SEQ_LEN, N_FEATURES), dtype=float)
    y = np.zeros((n_samples,), dtype=int)
    for i in range(n_samples):
        s = generate_sequence()
        X[i] = s
        y[i] = label_sequence(s)
    return X, y

# ---------------------------
# Build dataset
# ---------------------------
print("Generating synthetic dataset...")
X, y = generate_dataset(N_SAMPLES)
print("Class distribution (SAFE=0, CAUTION=1, IMMINENT=2):", np.bincount(y))

# One-hot labels
y_cat = utils.to_categorical(y, num_classes=3)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print("Training shape:", X_train.shape, " Test shape:", X_test.shape)

# ---------------------------
# LSTM model
# ---------------------------
def make_model(seq_len, n_features):
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.Masking(mask_value=0.0)(inp)
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(3, activation='softmax')(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = make_model(SEQ_LEN, N_FEATURES)
model.summary()

# Early stopping
es = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

# ---------------------------
# Train
# ---------------------------
print("Training model...")
history = model.fit(X_train, y_train, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es], verbose=2)

# ---------------------------
# Evaluate
# ---------------------------
print("Evaluating on test set...")
y_pred = model.predict(X_test)
y_test_idx = np.argmax(y_test, axis=1)
y_pred_idx = np.argmax(y_pred, axis=1)
print(classification_report(y_test_idx, y_pred_idx, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test_idx, y_pred_idx))

# ---------------------------
# Simulated realtime sliding-window inference
# ---------------------------
print("\n--- Starting realtime sliding-window simulation ---\n")

# Create an initial buffer (SEQ_LEN-1 previous timesteps)
buffer = np.zeros((SEQ_LEN-1, N_FEATURES), dtype=float)

def alert_action(pred_label, prob):
    if pred_label == 2:
        return f"IMMINENT ALERT 🚨 (p={prob:.2f}) — Evacuate now!"
    elif pred_label == 1:
        return f"CAUTION ⚠️ (p={prob:.2f}) — Monitor and prepare."
    else:
        return f"SAFE ✅ (p={prob:.2f}) — Normal operations."

# Simulate continuous stream of single timesteps
for step in range(60):  # simulate 60 timesteps ~ seconds/minutes depending on config
    # generate a new single timestep (not full sequence)
    new_t = generate_sequence()[ -1 ]  # reuse generator: take last row as new timestep
    # append to buffer to form a window
    window = np.vstack([buffer, new_t])
    assert window.shape == (SEQ_LEN, N_FEATURES)
    # normalize or scale if needed - here model trained on raw synthetic so keep same
    window_input = window.reshape(1, SEQ_LEN, N_FEATURES)
    pred = model.predict(window_input, verbose=0)[0]
    pred_idx = int(np.argmax(pred))
    pred_prob = float(np.max(pred))
    alert_txt = alert_action(pred_idx, pred_prob)

    # print the last few sensor values and alert
    print(f"Step {step+1:02d} | Last timestep Slope={new_t[0]:.2f}, Vib={new_t[1]:.2f}, Rain={new_t[2]:.2f} | Pred={pred_idx} p={pred_prob:.2f} -> {alert_txt}")

    # shift buffer
    buffer = np.vstack([buffer[1:], new_t])

    # placeholder: integrate alerting channel (SMS, webhook, siren) here
    # e.g., if pred_idx == 2 and pred_prob > 0.85: call send_sms(...) or trigger_relay(...)

    time.sleep(0.5)

print("\nSimulation finished.")


Generating synthetic dataset...
Class distribution (SAFE=0, CAUTION=1, IMMINENT=2): [ 312 2188  500]
Training shape: (2400, 30, 3)  Test shape: (600, 30, 3)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30, 3)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30, 3)     │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 30, 3)     │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 30)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 30, 64)    │     17,408 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 64)    │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32)        │        128 │ lstm_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,056 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │         99 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 31,107 (121.51 KB)

 Trainable params: 31,043 (121.26 KB)

 Non-trainable params: 64 (256.00 B)

Training model...
Epoch 1/20
34/34 - 13s - 368ms/step - accuracy: 0.7356 - loss: 0.6516 - val_accuracy: 0.7833 - val_loss: 0.5934
Epoch 2/20
34/34 - 2s - 51ms/step - accuracy: 0.7949 - loss: 0.4892 - val_accuracy: 0.7958 - val_loss: 0.5288
Epoch 3/20
34/34 - 2s - 50ms/step - accuracy: 0.8097 - loss: 0.4397 - val_accuracy: 0.8167 - val_loss: 0.4716
Epoch 4/20
34/34 - 3s - 83ms/step - accuracy: 0.8407 - loss: 0.3905 - val_accuracy: 0.8500 - val_loss: 0.4275
Epoch 5/20
34/34 - 4s - 117ms/step - accuracy: 0.8398 - loss: 0.3768 - val_accuracy: 0.8625 - val_loss: 0.3825
Epoch 6/20
34/34 - 2s - 72ms/step - accuracy: 0.8653 - loss: 0.3358 - val_accuracy: 0.8833 - val_loss: 0.3408
Epoch 7/20
34/34 - 3s - 74ms/step - accuracy: 0.8773 - loss: 0.3063 - val_accuracy: 0.8833 - val_loss: 0.3287
Epoch 8/20
34/34 - 2s - 55ms/step - accuracy: 0.8806 - loss: 0.2840 - val_accuracy: 0.8792 - val_loss: 0.2451
Epoch 9/20
34/34 - 3s - 88ms/step - accuracy: 0.9028 - loss: 0.2364 - val_accuracy: 0.9125 - val_lo